# 导入模块必要的包

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

# 1. 构建 U-Net

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = self.conv_block(1, 64)
        self.enc2 = self.conv_block(64, 128)
        self.bottleneck = self.conv_block(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = self.conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = self.conv_block(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(nn.MaxPool2d(2)(e1))
        b = self.bottleneck(nn.MaxPool2d(2)(e2))
        d1 = self.up1(b)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        return torch.sigmoid(self.final(d2))

# 2. 构建 MNIST 分割数据集（掩码 = 图像 > 0）

In [ ]:
class MNISTSeg(Dataset):
    def __init__(self):
        self.mnist = datasets.MNIST('data', train=True, download=True)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self): return len(self.mnist)

    def __getitem__(self, idx):
        img, _ = self.mnist[idx]
        img = self.transform(img)
        mask = (img > 0).float()  # 二值掩码
        return img, mask

# 3. 训练 U-Net

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
unet = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(unet.parameters(), lr=1e-3)

train_loader = DataLoader(MNISTSeg(), batch_size=32, shuffle=True)

for epoch in range(3):  # 极简训练
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        pred = unet(imgs)
        loss = criterion(pred, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

# 4. 可视化结果

In [ ]:
imgs, masks = next(iter(train_loader))
with torch.no_grad():
    out = unet(imgs[:4].to(device)).cpu()

fig, axs = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axs[0,i].imshow(imgs[i].squeeze(), cmap='gray')
    axs[1,i].imshow(out[i].squeeze(), cmap='gray')
    axs[0,i].axis('off'); axs[1,i].axis('off')
plt.show()